# Update employee skills after the last review

This notebook creates `employees_updated.json` without modifying the source `employees.json`.

For every employee it applies skill gains from activities with `status == "completed"` whose completion date is strictly later than `last_review_date`. Each gain is capped by the event's `max_level`. Activities are processed chronologically. Missing career goals are filled with the next grade in the employee's current role. Lead employees receive a `leadership_growth` or `role_mastery` goal with up to three focus skills that have a real skill gap and at least one eligible development event.

In [ ]:
from copy import deepcopy
from pathlib import Path
import csv
import json
from datetime import date

DATA_DIR = Path.cwd()
if not (DATA_DIR / "employees.json").exists():
    candidate = DATA_DIR / "case_1" / "career_quest_dataset"
    if (candidate / "employees.json").exists():
        DATA_DIR = candidate
    else:
        raise FileNotFoundError("Run the notebook from the dataset folder or the project root.")

EMPLOYEES_PATH = DATA_DIR / "employees.json"
EVENTS_PATH = DATA_DIR / "events.json"
SKILLS_PATH = DATA_DIR / "skills.json"
HISTORY_PATH = DATA_DIR / "activity_history.csv"
OUTPUT_PATH = DATA_DIR / "employees_updated.json"

print(f"Dataset folder: {DATA_DIR.resolve()}")

In [ ]:
with EMPLOYEES_PATH.open(encoding="utf-8") as file:
    employees_document = json.load(file)

with EVENTS_PATH.open(encoding="utf-8") as file:
    events_document = json.load(file)

with SKILLS_PATH.open(encoding="utf-8") as file:
    skills_document = json.load(file)

with HISTORY_PATH.open(encoding="utf-8", newline="") as file:
    activity_history = list(csv.DictReader(file))

employees = employees_document["employees"]
events = events_document["events"]
employees_by_id = {employee["employee_id"]: employee for employee in employees}
events_by_id = {event["event_id"]: event for event in events}
role_profiles = {(profile["role"], profile["grade"]): profile for profile in skills_document["role_profiles"]}

assert len(employees_by_id) == len(employees), "Duplicate employee_id detected"
assert len(events_by_id) == len(events), "Duplicate event_id detected"

unknown_employees = sorted({row["employee_id"] for row in activity_history} - employees_by_id.keys())
unknown_events = sorted({row["event_id"] for row in activity_history} - events_by_id.keys())
assert not unknown_employees, f"Unknown employees in history: {unknown_employees}"
assert not unknown_events, f"Unknown events in history: {unknown_events}"

print(f"Loaded {len(employees)} employees, {len(events)} events and {len(activity_history)} history rows.")

In [ ]:
updated_document = deepcopy(employees_document)
updated_employees_by_id = {employee["employee_id"]: employee for employee in updated_document["employees"]}

completed_rows = sorted(
    (row for row in activity_history if row["status"] == "completed"),
    key=lambda row: (date.fromisoformat(row["date"]), row["record_id"]),
)

applied_completions = 0
skill_changes = []

for row in completed_rows:
    employee = updated_employees_by_id[row["employee_id"]]
    completion_date = date.fromisoformat(row["date"])
    review_date = date.fromisoformat(employee["last_review_date"])

    if completion_date <= review_date:
        continue

    event = events_by_id[row["event_id"]]
    applied_completions += 1

    for development in event["develops_skills"]:
        skill_id = development["skill_id"]
        old_level = int(employee["skills"].get(skill_id, 0))
        new_level = min(old_level + int(development["gain"]), int(development["max_level"]))
        employee["skills"][skill_id] = new_level

        if new_level != old_level:
            skill_changes.append({
                "employee_id": employee["employee_id"],
                "event_id": event["event_id"],
                "completion_date": row["date"],
                "skill_id": skill_id,
                "old_level": old_level,
                "new_level": new_level,
            })

changed_employee_ids = {change["employee_id"] for change in skill_changes}
print(f"Applied {applied_completions} post-review completions.")
print(f"Changed {len(skill_changes)} skill values for {len(changed_employee_ids)} employees.")

next_grade = {"Junior": "Middle", "Middle": "Senior", "Senior": "Lead"}
filled_missing_goals = []
generated_lead_goals = []

for employee in updated_document["employees"]:
    if employee["career_goal"] is not None:
        continue

    if employee["grade"] != "Lead":
        employee["career_goal"] = {
            "target_role": employee["role"],
            "target_grade": next_grade[employee["grade"]],
        }
        filled_missing_goals.append(employee["employee_id"])
        continue

    profile = role_profiles[(employee["role"], "Lead")]
    critical_skills = set(profile["critical_skills"])
    skill_candidates = []

    for skill_id, required_level in profile["required_skills"].items():
        current_level = int(employee["skills"].get(skill_id, 0))
        gap = int(required_level) - current_level
        if gap <= 0:
            continue

        supporting_events = []
        for event in events:
            if event["mandatory"]:
                continue
            if employee["role"] not in event["target_roles"] or "Lead" not in event["target_grades"]:
                continue
            if any(employee["skills"].get(prerequisite_skill, 0) < minimum_level for prerequisite_skill, minimum_level in event["prerequisites"].items()):
                continue

            develops_skill = any(
                development["skill_id"] == skill_id and current_level < int(development["max_level"])
                for development in event["develops_skills"]
            )
            if develops_skill:
                supporting_events.append(event["event_id"])

        if supporting_events:
            skill_candidates.append({
                "skill_id": skill_id,
                "gap": gap,
                "critical": skill_id in critical_skills,
                "supporting_events": supporting_events,
            })

    skill_candidates.sort(
        key=lambda candidate: (
            not candidate["critical"],
            -candidate["gap"],
            -len(candidate["supporting_events"]),
            candidate["skill_id"],
        )
    )
    focus_skills = [candidate["skill_id"] for candidate in skill_candidates[:3]]
    has_critical_gap = any(candidate["critical"] for candidate in skill_candidates)

    employee["career_goal"] = {
        "goal_type": "leadership_growth" if has_critical_gap else "role_mastery",
        "target_role": employee["role"],
        "target_grade": "Lead",
        "focus_skills": focus_skills,
    }
    filled_missing_goals.append(employee["employee_id"])
    generated_lead_goals.append({
        "employee_id": employee["employee_id"],
        "goal": employee["career_goal"],
        "candidate_details": skill_candidates[:3],
    })

print(f"Filled {len(filled_missing_goals)} missing career goals.")
print(f"Generated {len(generated_lead_goals)} event-backed Lead development goals.")

In [ ]:
original_fields = [set(employee) for employee in employees]
updated_fields = [set(employee) for employee in updated_document["employees"]]
assert len(updated_document["employees"]) == len(employees)
assert original_fields == updated_fields, "Employee record schema changed"

for employee in updated_document["employees"]:
    assert employee["career_goal"] is not None, f"Missing career goal: {employee['employee_id']}"
    if employee["grade"] == "Lead" and employee["employee_id"] in filled_missing_goals:
        assert employee["career_goal"]["goal_type"] in {"leadership_growth", "role_mastery"}
        assert employee["career_goal"]["focus_skills"], f"No event-backed Lead focus skill: {employee['employee_id']}"
    for skill_id, level in employee["skills"].items():
        assert isinstance(level, int), f"Non-integer level: {employee['employee_id']} {skill_id}"
        assert 0 <= level <= 5, f"Level outside 0..5: {employee['employee_id']} {skill_id}={level}"

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    json.dump(updated_document, file, ensure_ascii=False, indent=2)
    file.write("\n")

print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
generated_lead_goals